# Generating RAG Answers

Importamos la verdad verdadera

In [1]:
import pandas as pd

df_ground_truth = pd.read_csv("data/ground_truth-new.csv")
ground_truth = df_ground_truth.to_dict(orient="records")

Ahora importamos los datos

In [2]:
from ingest import load_faq_data, build_index

documents = load_faq_data()

documents_llm = []

for doc in documents:
    if doc["course"] == "llm-zoomcamp":
        documents_llm.append(doc)

documents = documents_llm
index = build_index(documents)

In [3]:
doc_idx = {}

for doc in documents:
    doc_idx[doc["id"]] = doc

iniciamos el RAG

In [4]:
from dotenv import load_dotenv
from openai import OpenAI

load_dotenv()
openai_client = OpenAI()

In [5]:
from evaluation_utils import RAGWithUsage

assistant = RAGWithUsage(
    index=index,
    llm_client=openai_client,
)

Ahora lo corremos para una sola pregunta

In [6]:
rec = ground_truth[0]
question = rec["question"]

answer_llm = assistant.rag(question)
answer_llm

'Yes, you can still join. If you want a certificate, you need to submit your project while submissions are still being accepted.'

In [8]:
rec

{'question': 'Is it too late to join the course if I just found it?',
 'document': '74eb249bbf'}

In [9]:
assistant.total_cost()

0.00056475

In [10]:
doc_id = rec["document"]
original_doc = doc_idx[doc_id]
answer_orig = original_doc["answer"]

answer_orig

'Yes, but if you want to receive a certificate, you need to submit your project while we’re still accepting submissions.'

Luego la idea seria compara con un LLM qué tan diferentes con estas dos cosas

In [11]:
rag_result = {
    "question": question,
    "answer_llm": answer_llm,
    "answer_orig": answer_orig,
    "document": doc_id,
}

rag_result

{'question': 'Is it too late to join the course if I just found it?',
 'answer_llm': 'Yes, you can still join. If you want a certificate, you need to submit your project while submissions are still being accepted.',
 'answer_orig': 'Yes, but if you want to receive a certificate, you need to submit your project while we’re still accepting submissions.',
 'document': '74eb249bbf'}

Ahora lo ponemos todo junto en una funcion

In [12]:
def generate_rag_answer(rec):
    question = rec["question"]
    doc_id = rec["document"]
    original_doc = doc_idx[doc_id]

    answer_llm = assistant.rag(question)
    answer_orig = original_doc["answer"]

    result = {
        "question": question,
        "answer_llm": answer_llm,
        "answer_orig": answer_orig,
        "document": doc_id,
    }

    return result

In [13]:
answer_record = generate_rag_answer(ground_truth[0])
answer_record

{'question': 'Is it too late to join the course if I just found it?',
 'answer_llm': 'Yes, you can still join the course. If you want a certificate, make sure you submit your project while submissions are still open.',
 'answer_orig': 'Yes, but if you want to receive a certificate, you need to submit your project while we’re still accepting submissions.',
 'document': '74eb249bbf'}

In [15]:
from concurrent.futures import ThreadPoolExecutor
from evaluation_utils import map_progress

with ThreadPoolExecutor(max_workers=6) as pool:
    results = map_progress(pool, ground_truth, generate_rag_answer)

  0%|          | 0/515 [00:00<?, ?it/s]

In [16]:
answers = []

for answer_record in results:
    answers.append(answer_record)

In [17]:
assistant.total_cost()

0.5212829999999999

In [18]:
df_answers = pd.DataFrame(answers)
df_answers.to_csv("data/rag-answers-new.csv", index=False)

In [19]:
df_answers.head(10)

,question,answer_llm,answer_orig,document
0,Is it too late to join the course if I just fo...,"Yes, you can still join the course. If you wan...","Yes, but if you want to receive a certificate,...",74eb249bbf
1,Can I still enroll after the course has alread...,"Yes, you can still join after the course has s...","Yes, but if you want to receive a certificate,...",74eb249bbf
2,"If I join now, can I still get a certificate s...","Yes, if you join now you can still get a certi...","Yes, but if you want to receive a certificate,...",74eb249bbf
3,What do I need to do to qualify for the certif...,"To qualify for the certificate, you need to fi...","Yes, but if you want to receive a certificate,...",74eb249bbf
4,Does the project have to be submitted before s...,"Yes. To get the certificate, you need to submi...","Yes, but if you want to receive a certificate,...",74eb249bbf
5,Do I actually need a confirmation email after ...,You’re already accepted — you do **not** need ...,You don't need it. You're accepted. You can al...,977bf7786c
6,Can I start watching the LLM Zoomcamp lessons ...,Yes. You can start watching the LLM Zoomcamp l...,You don't need it. You're accepted. You can al...,977bf7786c
7,Is there any check against a registered list f...,No. It is **not checked against any registered...,You don't need it. You're accepted. You can al...,977bf7786c
8,What is the point of registering for LLM Zoomc...,Registration is just to gauge interest before ...,You don't need it. You're accepted. You can al...,977bf7786c
9,"If I signed up for LLM Zoomcamp, when should I...",You don’t need to wait for a confirmation emai...,You don't need it. You're accepted. You can al...,977bf7786c


# LLM as a Judge

In [20]:
import pandas as pd

df_answers = pd.read_csv("data/rag-answers-new.csv")
answers = df_answers.to_dict(orient="records")

In [33]:
answers

[{'question': 'Is it too late to join the course if I just found it?',
  'answer_llm': 'Yes, you can still join the course. If you want to receive a certificate, make sure you submit your project while submissions are still being accepted.',
  'answer_orig': 'Yes, but if you want to receive a certificate, you need to submit your project while we’re still accepting submissions.',
  'document': '74eb249bbf'},
 {'question': 'Can I still enroll after the course has already started?',
  'answer_llm': 'Yes, you can still join after the course has started. If you want a certificate, you need to submit your project while the course is still accepting submissions.',
  'answer_orig': 'Yes, but if you want to receive a certificate, you need to submit your project while we’re still accepting submissions.',
  'document': '74eb249bbf'},
 {'question': 'If I join now, can I still get a certificate somehow?',
  'answer_llm': 'Yes, if you join now you can still get a certificate, but only if you finish 

In [21]:
from pydantic import BaseModel, Field
from typing import Literal

class AnswerEvaluation(BaseModel):
    reasoning: str = Field(
        description="Reasoning about the quality of the answer."
    )
    score: Literal["good", "bad"] = Field(
        description="'good' if the answer is correct and complete, 'bad' otherwise."
    )

In [22]:
aqa_judge_instructions = """
You are an expert evaluator. You will be given:
1. A question from a student
2. The original answer from the FAQ (ground truth)
3. An answer generated by an AI assistant

Your task is to decide if the AI answer is semantically equivalent to
the original answer.

Rules:
- The AI answer does NOT need to be word-for-word identical
- It should convey the same key information
- Extra detail is fine as long as the core answer is correct
- Mark 'bad' only if the AI answer is wrong or misses the key point

Be fair and focus on correctness, not style.
""".strip()

In [23]:
aqa_judge_prompt = """
Question:
{question}

Original Answer (ground truth):
{answer_orig}

AI Answer:
{answer_llm}
""".strip()

In [24]:
rec = answers[0]

In [25]:
prompt = aqa_judge_prompt.format(
    question=rec["question"],
    answer_orig=rec["answer_orig"],
    answer_llm=rec["answer_llm"]
)

In [27]:
from evaluation_utils import llm_structured_retry

eval_result, usage = llm_structured_retry(
    openai_client,
    aqa_judge_instructions,
    prompt,
    AnswerEvaluation,
)

eval_result

AnswerEvaluation(reasoning='The AI answer preserves the key meaning of the ground truth: it says it is still possible to join, and that receiving a certificate requires submitting the project before submissions close. This is semantically equivalent.', score='good')

In [29]:
from evaluation_utils import calc_price
calc_price(usage)

{'input_cost': 0.00022125,
 'output_cost': 0.000252,
 'total_cost': 0.00047325000000000004}

In [30]:
def evaluate_aqa(question, answer_orig, answer_llm, model="gpt-5.4-mini"):
    prompt = aqa_judge_prompt.format(
        question=question,
        answer_orig=answer_orig,
        answer_llm=answer_llm
    )

    result, usage = llm_structured_retry(
        openai_client,
        aqa_judge_instructions,
        prompt,
        AnswerEvaluation,
        model=model,
    )

    return result, usage

In [31]:
eval_result, usage = evaluate_aqa(
    question=rec["question"],
    answer_orig=rec["answer_orig"],
    answer_llm=rec["answer_llm"]
)

eval_result

AnswerEvaluation(reasoning='The AI answer preserves the core meaning of the ground truth: it is not too late to join, but receiving a certificate requires submitting the project before submissions close. This is semantically equivalent.', score='good')

In [32]:
def judge_record(rec):
    eval_result, usage = evaluate_aqa(
        question=rec["question"],
        answer_orig=rec["answer_orig"],
        answer_llm=rec["answer_llm"]
    )

    result = {
        "question": rec["question"],
        "document": rec["document"],
        "score": eval_result.score,
        "reasoning": eval_result.reasoning,
    }

    return result, usage

In [ ]:
from concurrent.futures import ThreadPoolExecutor

with ThreadPoolExecutor(max_workers=6) as pool:
    results = map_progress(pool, answers, judge_record)

Despues de esto evalua que tan bien hizo la performans

# Agent Evaluation

Primero cargamos la verdad verdadera

In [2]:
import pandas as pd

df_ground_truth = pd.read_csv("data/ground_truth-new.csv")
ground_truth = df_ground_truth.to_dict(orient="records")
ground_truth[0]

{'question': 'Is it too late to join the course if I just found it?',
 'document': '74eb249bbf'}

Ahora cargamos los documentos de los cuales se busca una respuesta

In [3]:
from ingest import load_faq_data, build_index

documents = load_faq_data()

documents_llm = []

for doc in documents:
    if doc["course"] == "llm-zoomcamp":
        documents_llm.append(doc)

documents = documents_llm
index = build_index(documents)

A estos documentos se les asigna una ID

In [4]:
doc_idx = {}

for doc in documents:
    doc_idx[doc["id"]] = doc

In [6]:
from dotenv import load_dotenv
from openai import OpenAI
from toyaikit.llm import OpenAIClient

load_dotenv()
openai_client = OpenAI()

Definimos la funcion de busqueda

In [7]:
def search(query: str) -> list[dict]:
    """
    Search the FAQ database for entries matching the given query.
    """
    return index.search(
        query,
        num_results=5,
        boost_dict={"question": 1.0, "answer": 2.0, "section": 0.1},
        filter_dict={"course": "llm-zoomcamp"}
    )

In [8]:
from toyaikit.tools import Tools
from toyaikit.chat.runners import OpenAIResponsesRunner

agent_tools = Tools()
agent_tools.add_tool(search)

instructions = """
You're a course teaching assistant. Answer student questions based on
the FAQ search results. Use the search tool before answering.
""".strip()

runner = OpenAIResponsesRunner(
    tools=agent_tools,
    developer_prompt=instructions,
    llm_client=OpenAIClient(model="gpt-5.4-mini")
)

Vamos a correrlo para un valor de la verdad verdadera

En el loop es cuando se le dice al agente: te doy una pregunta y decidi si quieres usar las herramientas que te doy o si quieres directamente respodnerme. En este caso si o si debe usar search la primera vez porques se lo pedimos

In [9]:
rec = ground_truth[0]

result = runner.loop(prompt=rec["question"])

In [11]:
result.all_messages

[EasyInputMessage(content="You're a course teaching assistant. Answer student questions based on\nthe FAQ search results. Use the search tool before answering.", role='developer', phase=None, type=None),
 EasyInputMessage(content='Is it too late to join the course if I just found it?', role='user', phase=None, type=None),
 ResponseFunctionToolCall(arguments='{"query":"too late to join course just found it late enrollment"}', call_id='call_Z8RXG7OAXHgMHUUciIgr4Nb2', name='search', type='function_call', id='fc_0d2bb44550cc99e3006a466999b41081939707c52657e7deac', namespace=None, status='completed'),
 {'type': 'function_call_output',
  'call_id': 'call_Z8RXG7OAXHgMHUUciIgr4Nb2',
  'output': '[\n  {\n    "id": "74eb249bbf",\n    "course": "llm-zoomcamp",\n    "section": "General Course-Related Questions",\n    "question": "I just discovered the course. Can I still join?",\n    "answer": "Yes, but if you want to receive a certificate, you need to submit your project while we\\u2019re still a

Si quieremos sacas las toolcalls

In [12]:
def extract_tool_calls(messages):
    tool_calls = []

    for message in messages:
        if isinstance(message, dict):
            continue

        if message.type == "function_call":
            tool_calls.append({
                "name": message.name,
                "arguments": message.arguments,
            })

    return tool_calls

In [13]:
tool_calls = extract_tool_calls(result.all_messages)

tool_calls

[{'name': 'search',
  'arguments': '{"query":"too late to join course just found it late enrollment"}'}]

In [14]:
doc_id = rec["document"]
original_doc = doc_idx[doc_id]
answer_orig = original_doc["answer"]

Ahora guardamos todo esto en un dic que se llame agent_result para despues evaluar su eficiencia

In [15]:
agent_result = {
    "question": rec["question"],
    "answer_agent": result.last_message,
    "answer_orig": answer_orig,
    "tool_calls": tool_calls,
    "cost": result.cost.total_cost,
    "document": doc_id,
}

agent_result

{'question': 'Is it too late to join the course if I just found it?',
 'answer_agent': 'Yes — you can still join if you just discovered the course.\n\nIf you want a certificate, make sure you submit your project while submissions are still open.',
 'answer_orig': 'Yes, but if you want to receive a certificate, you need to submit your project while we’re still accepting submissions.',
 'tool_calls': [{'name': 'search',
   'arguments': '{"query":"too late to join course just found it late enrollment"}'}],
 'cost': Decimal('0.00091575'),
 'document': '74eb249bbf'}

A todo este proceso de dado un rec generamos la respuesta del llm y la verdad verdadera lo ponermos en una funcion

In [16]:
def generate_agent_answer(rec):
    doc_id = rec["document"]
    original_doc = doc_idx[doc_id]

    result = runner.loop(prompt=rec["question"])

    tool_calls = extract_tool_calls(result.all_messages)

    answer_record = {
        "question": rec["question"],
        "answer_agent": result.last_message,
        "answer_orig": original_doc["answer"],
        "tool_calls": tool_calls,
        "cost": result.cost.total_cost,
        "document": doc_id,
    }

    return answer_record

Generamos varias agent_answe en paralelo

In [17]:
from concurrent.futures import ThreadPoolExecutor
from evaluation_utils import map_progress

with ThreadPoolExecutor(max_workers=6) as pool:
    agent_answers = map_progress(pool, ground_truth[:50], generate_agent_answer)

  0%|          | 0/50 [00:00<?, ?it/s]

Ahora las guardamos en un dataframe

In [18]:
df_agent = pd.DataFrame(agent_answers)

In [19]:
df_agent["cost"].sum()

Decimal('0.06384000')

Guardamos los resultados

In [20]:
df_agent.to_csv("data/agent-answers.csv", index=False)

Volvemos a cargar esto

In [21]:
df_agent = pd.read_csv("data/agent-answers.csv")
agent_answers = df_agent.to_dict(orient="records")

In [26]:
agent_answers[0]

{'question': 'Is it too late to join the course if I just found it?',
 'answer_agent': 'Yes — it’s not too late to join. If you just discovered the course, you can still start learning and participating.\n\nOne important note: if you want a certificate, you’ll need to submit your project while submissions are still being accepted.',
 'answer_orig': 'Yes, but if you want to receive a certificate, you need to submit your project while we’re still accepting submissions.',
 'tool_calls': '[{\'name\': \'search\', \'arguments\': \'{"query":"too late to join course just found it enrollment late join"}\'}]',
 'cost': 0.000993,
 'document': '74eb249bbf'}

Ahora que tenemos todo un historial de preguntas, respuestas, respuestas correctos y cantidad de calls. Generamos un objeto para ver si cada una cumple con algunas caracteristicas

In [22]:
from pydantic import BaseModel, Field
from typing import Literal

class AgentEvaluation(BaseModel):
    answer_reasoning: str = Field(
        description="Reasoning about whether the final answer is correct."
    )
    answer_score: Literal["good", "bad"] = Field(
        description="'good' if the final answer matches the original answer."
    )
    trajectory_reasoning: str = Field(
        description="Reasoning about whether the tool calls were useful."
    )
    trajectory_score: Literal["good", "bad"] = Field(
        description="'good' if the tool calls were reasonable for the question."
    )

In [23]:
agent_judge_instructions = """
You are an expert evaluator. You will be given:
1. A question from a student
2. The original answer from the FAQ (ground truth)
3. An answer generated by an AI agent
4. The tool calls made by the agent

Evaluate two things:

Answer quality:
- Does the agent answer match the original answer?
- It does not need to be word-for-word identical.
- It should contain the same key information.

Trajectory quality:
- Were the search queries relevant to the question?
- Did the queries include important keywords from the question?
- Did the agent avoid duplicate or unnecessary tool calls?
- If it made multiple searches, did the later searches refine the query?
- Was the number of search calls reasonable? Usually 1 is enough, 2-3
  can be okay, and more than 3 needs a clear reason.
- Did the tool calls support the final answer?

Mark answer_score as 'good' if the final answer is correct.
Mark trajectory_score as 'good' if the tool calls were reasonable.
""".strip()

agent_judge_prompt = """
Question:
{question}

Original Answer (ground truth):
{answer_orig}

Agent Answer:
{answer_agent}

Tool Calls:
{tool_calls}
""".strip()

Definimos la faucnion juez

In [28]:
import ast
import json
from evaluation_utils import calc_total_price, llm_structured_retry

def evaluate_agent_answer(rec, model="gpt-5.4-mini"):
    tool_calls = rec["tool_calls"]

    if isinstance(tool_calls, str):
        try:
            tool_calls = json.loads(tool_calls)
        except json.JSONDecodeError:
            tool_calls = ast.literal_eval(tool_calls)

    prompt = agent_judge_prompt.format(
        question=rec["question"],
        answer_orig=rec["answer_orig"],
        answer_agent=rec["answer_agent"],
        tool_calls=json.dumps(tool_calls, indent=2),
    )

    result, usage = llm_structured_retry(
        openai_client,
        agent_judge_instructions,
        prompt,
        AgentEvaluation,
        model=model,
    )

    return result, usage

In [29]:
agent_eval, usage = evaluate_agent_answer(agent_answers[0])

agent_eval

AgentEvaluation(answer_reasoning='The agent’s answer matches the ground truth. It correctly says it is not too late to join the course and includes the key condition for getting a certificate: the project must be submitted while submissions are still being accepted.', answer_score='good', trajectory_reasoning='The search query was relevant to the question and included core keywords about joining late and finding the course just now. Only one search call was made, which is reasonable for this simple FAQ-style question. The tool call supported the final answer.', trajectory_score='good')

Ahora queda correrlo para todas las respuestas

In [30]:
def judge_agent_record(rec):
    agent_eval, usage = evaluate_agent_answer(rec)

    result = {
        "question": rec["question"],
        "document": rec["document"],
        "answer_score": agent_eval.answer_score,
        "answer_reasoning": agent_eval.answer_reasoning,
        "trajectory_score": agent_eval.trajectory_score,
        "trajectory_reasoning": agent_eval.trajectory_reasoning,
    }

    return result, usage

In [ ]:
with ThreadPoolExecutor(max_workers=6) as pool:
    results = map_progress(pool, agent_answers, judge_agent_record)

In [ ]:
agent_evaluations = []
usages = []

for evaluation, usage in results:
    agent_evaluations.append(evaluation)
    usages.append(usage)

Y luego se evalua